In [3]:
import os
from dotenv import load_dotenv
from unstructured.partition.pdf import partition_pdf

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
LANGCHAIN_API_KEY = os.getenv("LANGSMITH_API_KEY")
LANGCHAIN_TRACING_V2 = "true"


output_path = "../output/"
file_path = "../content/EINSTEIN_264_PLANOS.pdf"

In [9]:
from unstructured.partition.pdf import partition_pdf

# Reference: https://docs.unstructured.io/open-source/core-functionality/chunking
chunks = partition_pdf(
    filename=file_path,
    infer_table_structure=True,            # extract tables
    strategy="fast",                     # mandatory to infer tables

    extract_image_block_types=["Image", "Table"],   # Add 'Table' to list to extract image of tables
    # image_output_dir_path=output_path,   # if None, images and tables will saved in base64

    extract_image_block_to_payload=True,   # if true, will extract base64 for API usage

    chunking_strategy="basic",          # or 'basic'
    max_characters=20000,                  # defaults to 500
    combine_text_under_n_chars=4000,       # defaults to 0
    pdfminer_line_margin=0.5,             # Ajustar según necesidad
    pdfminer_char_margin=2.0,             # Aumentado para mejor detección de caracteres en planos
    pdfminer_word_margin=0.1,

    # extract_images_in_pdf=True,          # deprecated
)


CropBox missing from /Page, defaulting to MediaBox


CropBox missing from /Page, defaulting to MediaBox


In [14]:
chunks[0].metadata.orig_elements

In [15]:
texts = []
for chunk in chunks:
    texts.append(chunk)

In [25]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from base64 import b64decode
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

docs = chunks



def parse_docs(docs):
    context_text = ""
    for doc in docs:
        for element in doc.metadata.orig_elements:
            if "Table" in str(type(element)):
                context_text += element.metadata.text_as_html
            else:
                context_text += element.text
    return context_text

# Definir el template
template = """Responde la pregunta basada unicamente en el siguiente contexto, que puede incluir texto y tablas.
Contexto: {context}
Pregunta: {question}"""

prompt = ChatPromptTemplate.from_template(template)
model = ChatGroq(temperature=0.5, model="gemma2-9b-it")
chain = prompt | model | StrOutputParser()

# Preparar los datos
context = parse_docs(docs)


prompt_data = {
    "context": context,  # Asegúrate de formatear el contexto apropiadamente
    "question": "cual es el factor de constructibilidad y superficie de constructibilidad declarados en el plano?"
}

# Hacer la llamada
response = chain.invoke(prompt_data)
print(response)

El texto proporcionado no menciona el factor de constructibilidad ni la superficie de constructibilidad. 



